# Nino Fine-Tuning (Open Images + optional custom photos)

This notebook trains a fresh EfficientDet-Lite0 object detector for the Nino assistive app. The training set is pulled automatically from the Open Images v7 dataset (already labeled, no manual work). Optionally you can merge your own labeled PascalVOC photos.

The class list is a **compact 13-class classroom prototype set**: the people, seating/desks, room boundaries, gadgets, books and hazards a blind student actually meets inside a classroom. Keeping the class list small (12-13) trains faster, downloads far less data, and detects exactly these few objects much more accurately than the generic 90-class COCO model the app originally shipped with.

## How to run (no restart needed)
1. **Runtime -> Change runtime type -> T4 GPU** (recommended, much faster).
2. Press **Run all** and let it run. Do not close or background this tab; disable your laptop sleep.
3. The notebook switches the shell to Python 3.11 (required by the trainer) but never restarts the Colab kernel.

**No GPU? Your laptop does NOT need one** - Colab runs on Google's servers, so integrated graphics is fine. And if Colab assigns you a CPU-only runtime anyway, the config cell detects it automatically and switches to lighter training settings (fewer samples per class, fewer epochs, 192px images) so training still finishes.

**T4 note:** the install step pulls `tensorflow[and-cuda]`, because pip TensorFlow no longer bundles the NVIDIA CUDA libraries - without it TensorFlow silently falls back to CPU even when a T4 is attached. A dedicated GPU-check cell then verifies that TensorFlow really sees the T4 and stops loudly if it does not, so you never waste hours training on CPU by accident.

## Timing
- Steps 1-2 (switch to Python 3.11, install packages incl. CUDA-enabled TensorFlow): ~5-8 min
- Step 4 (class + GPU check): ~30 sec
- Step 5 (download images from Open Images): 5-20 min (15-class classroom/college set)
- Step 7 (train + export model): ~10-25 min on T4 GPU (batch 16), under an hour on CPU
- Step 8: downloads `nino_model.zip` automatically

## Troubleshooting
- A warning like `WARN SomeClass : 0 samples` is fine - that class just gets skipped.
- If the GPU-check cell prints `A GPU is attached but TensorFlow cannot see it`, re-run the install cell (make sure `tensorflow[and-cuda]` is in that pip line) and run the check again. Do NOT continue to training in that state - it would silently run on CPU for hours.
- While Step 7 runs you can open a fresh cell and run `!nvidia-smi` a few times; GPU utilization should spike while training epochs are progressing. If it stays at 0%, TensorFlow is on CPU - stop and fix the install first.
- If you see `No matching distribution found for pandas222` (or any odd package name), you are running a stale or hand-edited copy - the real package is `pandas`. Reupload this notebook from the repo and run all.
- If you see `Key backend: ... is not a valid value for backend`, you are running an outdated copy. Close the tab, reopen the latest Nino_Trainer.ipynb from the repo, and run all again.
- Every step script forces the matplotlib backend to `Agg` both inside the script and on the shell command line, so the notebook backend is never picked up in the training subprocesses.

In [ ]:
# --- 0. Switch the shell's python3 to Python 3.11 (required by mediapipe-model-maker) ---
# IMPORTANT: no runtime restart is needed. The notebook kernel stays as-is;
# every heavy step below runs as a subprocess under /usr/bin/python3 (now 3.11).
!sudo apt-get update -y -qq
!if ! command -v python3.11 >/dev/null 2>&1; then sudo add-apt-repository -y ppa:deadsnakes/ppa && sudo apt-get update -y -qq; fi
!sudo apt-get install -y -qq python3.11 python3.11-distutils
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!sudo update-alternatives --set python3 /usr/bin/python3.11
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py && /usr/bin/python3 /tmp/get-pip.py -q
!/usr/bin/python3 --version

In [ ]:
!/usr/bin/python3 --version
!/usr/bin/python3 -m pip --version

In [ ]:
!/usr/bin/python3 -m pip install -q --upgrade pip
# mediapipe-model-maker needs tensorflow<2.16, and pip TensorFlow no longer bundles
# the NVIDIA CUDA libraries. Installing plain mediapipe-model-maker gives you a
# CPU-only TF build -> the T4 sits idle. So install tensorflow[and-cuda] 2.15 with
# keras<3 in the SAME pip resolve so all constraints stay consistent.
!/usr/bin/python3 -m pip install -q 'keras<3.0.0' 'tensorflow[and-cuda]~=2.15.1' mediapipe-model-maker
# numpy<2 guard: TF 2.15 is built against the numpy 1.x ABI.
!/usr/bin/python3 -m pip install -q fiftyone opencv-python-headless pandas 'numpy<2'


In [ ]:
%%writefile /content/step0_gpu_check.py
import os
os.environ["MPLBACKEND"] = "Agg"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import shutil

has_nvidia = shutil.which("nvidia-smi") is not None
if has_nvidia:
    os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPUs visible to TensorFlow:", gpus if gpus else "NONE (CPU only)")

if has_nvidia and not gpus:
    raise SystemExit(
        "ERROR: a GPU is attached but TensorFlow cannot see it. "
        "Re-run the install cell above, then re-run this cell. "
        "Do not start training until this check passes - otherwise it runs on CPU for hours."
    )
if gpus:
    print("GPU ready - training will run on CUDA.")
else:
    print("No GPU on this runtime - continuing with the lighter CPU training profile.")

In [ ]:
!/usr/bin/python3 -c "import pandas, cv2, fiftyone; print('install OK | pandas', pandas.__version__, '| opencv', cv2.__version__)"
!MPLBACKEND=Agg /usr/bin/python3 /content/step0_gpu_check.py

## 1. Choose your classes

Edit the `CLASSES` list in the next cell to add or remove class names. Names must match the Open Images dataset exactly; the step after this cell checks them for you.

The default set is a **compact 13-class classroom prototype list**: people, seating/desks, room boundaries (door/window), gadgets, books and the fan hazard a blind student actually meets. Keeping the count small makes training much faster and improves detection of exactly these objects.

To cover more of your campus, append exact Open Images names such as `Houseplant`, `Umbrella`, `Wheelchair`, `Escalator`, `Backpack`, or `Bicycle` - but remember every extra class adds download + training time.

In [ ]:
import json
import shutil

# Finalised 13-class classroom prototype set - exactly what a blind student meets
# inside a classroom for the capstone demo. Every name is a valid boxable Open Images
# v7 class (checked by the validation cell below). Fewer classes = faster training,
# less data downloaded, and more accurate detection of these specific objects.
# NOTE: 'Fan' is not boxable in OIDv7 - use 'Mechanical fan'. People are covered by
# 'Person' (no 'Teacher' class exists), and glass/windows are covered by 'Window'.
CLASSES = [
    # people - the most important obstacle (covers teachers & students)
    "Person",
    # classroom furniture & seating obstacles
    "Chair", "Table", "Desk", "Bench",
    # room boundaries & glass
    "Door", "Window",
    # spinning body-level hazard
    "Mechanical fan",
    # gadgets, books & boards left around
    "Laptop", "Mobile phone", "Book", "Whiteboard",
    # floor-level obstacle
    "Backpack",
]

gpu = shutil.which("nvidia-smi") is not None
mode = "GPU" if gpu else "CPU"
print("Hardware detected:", mode)

config = {
    "classes": CLASSES,
    "max_samples_per_class": 120 if gpu else 25,
    "val_split": 0.10,
    "epochs": 30 if gpu else 6,
    # 16 comfortably fits EfficientDet-Lite0 @320px in a T4's 16 GB and keeps it fed;
    # batch 8 would leave the GPU idle half the time. CPU profile stays at 8.
    "batch_size": 16 if gpu else 8,
    "image_size": 320 if gpu else 192,
    "seed": 42,
    "skip_custom_data": True,       # set False if you also want to merge your own PascalVOC photos
}

with open("/content/nino_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Training profile:", mode, "|", config["max_samples_per_class"], "imgs/class,",
      config["epochs"], "epochs,", config["image_size"], "px, batch", config["batch_size"])
print(len(CLASSES), "classes configured")
print(CLASSES)

## 2. Validate class names

Run the two cells below. The first writes the validation script, the second executes it under Python 3.11.

In [ ]:
%%writefile /content/step1_validate.py
import os
os.environ["MPLBACKEND"] = "Agg"
import json
import subprocess
import pandas as pd

cfg = json.load(open("/content/nino_config.json"))
classes = cfg["classes"]

subprocess.run(
    ["curl", "-sL",
     "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions.csv",
     "-o", "/content/oid_classes.csv"],
    check=True,
)
df = pd.read_csv("/content/oid_classes.csv", names=["id", "name"])
valid = set(df["name"])

missing = [c for c in classes if c not in valid]
print(len(classes), "classes requested")
if missing:
    print("INVALID class names - fix CLASSES in the config cell above:")
    for m in missing:
        print("  -", m)
    raise SystemExit(1)
print("All class names are valid.")

In [ ]:
!MPLBACKEND=Agg /usr/bin/python3 /content/step1_validate.py

## 3. Download the Open Images data

This downloads up to `max_samples_per_class` labeled images per class (already annotated, no labeling work) and saves a deduplicated snapshot. It is the longest download step - keep the tab open.

In [ ]:
%%writefile /content/step2_download.py
import os
os.environ["MPLBACKEND"] = "Agg"
import json
import fiftyone as fo
import fiftyone.zoo as foz

cfg = json.load(open("/content/nino_config.json"))
classes = cfg["classes"]
max_samples = cfg["max_samples_per_class"]
seed = cfg["seed"]

sets = []
for c in classes:
    name = "nino_" + c.lower().replace(" ", "_")
    try:
        ds = foz.load_zoo_dataset(
            "open-images-v7",
            split="train",
            label_types=["detections"],
            classes=[c],
            max_samples=max_samples,
            shuffle=True,
            seed=seed,
            dataset_name=name,
        )
        if len(ds) == 0:
            print("WARN", c, ": 0 samples - skipping")
        else:
            sets.append(ds)
            print(c, ":", len(ds), "samples")
    except Exception as e:
        print("SKIP", c, ":", e)

if not sets:
    raise RuntimeError("No class data could be downloaded. Check class names and network, then rerun.")

dataset = sets[0].concat(sets[1:])

seen, keep = set(), []
for sample in dataset:
    if sample.filepath not in seen:
        seen.add(sample.filepath)
        keep.append(sample.id)
dataset = dataset.select(keep)

print("Total unique images:", len(dataset))
print("Downloading image files and saving a dataset snapshot...")
dataset.export("/content/nino_ds", dataset_type=fo.types.FiftyOneDataset)
print("Saved to /content/nino_ds")

In [ ]:
!MPLBACKEND=Agg /usr/bin/python3 /content/step2_download.py

## 4. (Optional) Merge your own photos

If `skip_custom_data` in the config cell is `False`, the next cell lets you upload a PascalVOC zip (folders `Annotations` + `JPEGImages`) of photos you labeled yourself. With `skip_custom_data=True` (default) this cell is skipped automatically.

In [ ]:
import json
import os
import zipfile
from google.colab import files

cfg = json.load(open("/content/nino_config.json"))
custom_dir = "/content/custom_voc"

if cfg["skip_custom_data"]:
    print("skip_custom_data=True - no custom photos needed, using Open Images data only.")
else:
    os.makedirs(custom_dir, exist_ok=True)
    print("Upload your PascalVOC zip (folders Annotations + JPEGImages):")
    up = files.upload()
    if up:
        zpath = list(up.keys())[0]
        with zipfile.ZipFile(zpath) as z:
            z.extractall(custom_dir)
        print("Extracted to", custom_dir)
        print(os.listdir(custom_dir))

Split into train/val and export PascalVOC for the trainer:

In [ ]:
%%writefile /content/step3_merge_export.py
import os
os.environ["MPLBACKEND"] = "Agg"
import json
import random
import fiftyone as fo

cfg = json.load(open("/content/nino_config.json"))
classes = cfg["classes"]
val_split = cfg["val_split"]
seed = cfg["seed"]

dataset = fo.Dataset.from_dir("/content/nino_ds", dataset_type=fo.types.FiftyOneDataset)
print("Loaded", len(dataset), "Open Images samples")

annot_dir = "/content/custom_voc/Annotations"
if os.path.isdir(annot_dir):
    custom_ds = fo.Dataset.from_dir("/content/custom_voc", dataset_type=fo.types.VOCDetectionDataset)
    dataset = dataset.concat(custom_ds)
    seen, keep = set(), []
    for sample in dataset:
        if sample.filepath not in seen:
            seen.add(sample.filepath)
            keep.append(sample.id)
    dataset = dataset.select(keep)
    print("Merged custom photos, total", len(dataset))
else:
    print("No custom data found - continuing with Open Images only.")

ids = list(dataset.values("id"))
random.seed(seed)
random.shuffle(ids)
n_val = max(1, int(len(ids) * val_split))
val_ids = set(ids[:n_val])
train_ids = set(ids[n_val:])

train_view = dataset.select(train_ids)
val_view = dataset.select(val_ids)

train_view.export(
    "/content/voc_train", dataset_type=fo.types.VOCDetectionDataset,
    label_field="detections", classes=classes,
)
val_view.export(
    "/content/voc_val", dataset_type=fo.types.VOCDetectionDataset,
    label_field="detections", classes=classes,
)
print("Exported VOC: /content/voc_train | /content/voc_val")

In [ ]:
!MPLBACKEND=Agg /usr/bin/python3 /content/step3_merge_export.py

## 5. Train object detector

This is the long step. On a T4 with the GPU profile (batch 16) expect roughly 25-60 minutes. It trains a MobileNet-based SSD detector on your class set and exports:
- `/content/detect.tflite` - the model the Nino app will run
- `/content/labels.txt` - the class list (line 0 = class 0, so the app's `labelOffset` becomes 0)

**Offset note:** this mediapipe export labels classes from index 0 (`labels.txt` line 0 = class 0). The Nino app auto-detects this, so installing these two files is all it takes to switch from the generic COCO model to this focused classroom model.

In [ ]:
%%writefile /content/step4_train.py
import os
os.environ["MPLBACKEND"] = "Agg"
# Must be set BEFORE tensorflow/model_maker imports: lets TF grow GPU memory on
# demand instead of grabbing all 16 GB up front.
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
import glob
import json
import shutil
import tensorflow as tf
from mediapipe_model_maker import object_detector

cfg = json.load(open("/content/nino_config.json"))
epochs = cfg["epochs"]
batch_size = cfg["batch_size"]
image_size = cfg["image_size"]

# Fail fast if the T4 is attached but TensorFlow ended up on CPU anyway.
gpus = tf.config.list_physical_devices("GPU")
device = "CUDA GPU" if gpus else "CPU"
print("TensorFlow", tf.__version__, ": training device:", gpus if gpus else "[CPU only]")
if shutil.which("nvidia-smi") is not None and not gpus:
    raise RuntimeError(
        "A GPU is attached but TensorFlow cannot use it. Re-run the install cell "
        "(it installs tensorflow[and-cuda]) and rerun this step - CPU would take many hours."
    )

# --- Restructure fiftyone VOC export to match mediapipe expected layout ---
# mediapipe expects: <data_dir>/images/*.jpg + <data_dir>/Annotations/*.xml
# fiftyone exports: <data_dir>/*.jpg + <data_dir>/Annotations/*.xml (flat)
for voc_dir in ["/content/voc_train", "/content/voc_val"]:
    img_dir = os.path.join(voc_dir, "images")
    os.makedirs(img_dir, exist_ok=True)
    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        for f in glob.glob(os.path.join(voc_dir, ext)):
            shutil.move(f, os.path.join(img_dir, os.path.basename(f)))
    print(voc_dir, ":", len(os.listdir(img_dir)), "images,", len(glob.glob(os.path.join(voc_dir, "Annotations", "*.xml"))), "annotations")

# --- Select model based on image size from config ---
if image_size >= 384:
    supported_model = object_detector.SupportedModels.MOBILENET_MULTI_AVG_I384
elif image_size >= 320:
    supported_model = object_detector.SupportedModels.MOBILENET_V2_I320
else:
    supported_model = object_detector.SupportedModels.MOBILENET_V2
print("Using model:", supported_model.name, "|", image_size, "px")

train_data = object_detector.Dataset.from_pascal_voc_folder("/content/voc_train")
val_data = object_detector.Dataset.from_pascal_voc_folder("/content/voc_val")
print("train samples:", len(train_data), "| val samples:", len(val_data))

hparams = object_detector.HParams(
    epochs=epochs,
    batch_size=batch_size,
    export_dir="/content/nino_model_export",
)
options = object_detector.ObjectDetectorOptions(
    supported_model=supported_model,
    hparams=hparams,
)

print("Training on", device, "| batch", batch_size, "|", image_size, "px |", epochs, "epochs (this is the long step)")
model = object_detector.ObjectDetector.create(train_data, val_data, options)
loss, coco_metrics = model.evaluate(val_data)
print("Evaluation loss:", loss, "| COCO metrics:", coco_metrics)

os.makedirs("/content/nino_model_export", exist_ok=True)
model.export_model(model_name="detect.tflite")

label_names = train_data.label_names
with open("/content/labels.txt", "w") as f:
    for name in label_names:
        f.write(name + "\n")

print("Trained labels (%d):" % len(label_names))
print(label_names)
tflite_path = "/content/nino_model_export/detect.tflite"
if os.path.exists(tflite_path):
    shutil.copy(tflite_path, "/content/detect.tflite")
    print("Model saved to /content/detect.tflite")
else:
    print("ERROR: TFLite export not found at", tflite_path)
    print("Export dir contents:", os.listdir("/content/nino_model_export"))

In [ ]:
!MPLBACKEND=Agg /usr/bin/python3 /content/step4_train.py

## 6. Download the trained model

In [ ]:
!zip -r -j /content/nino_model.zip /content/detect.tflite /content/labels.txt
from google.colab import files
files.download("/content/nino_model.zip")
print("Downloaded. Send me nino_model.zip and I will install it into the Nino app.")

## Done!

`nino_model.zip` contains `detect.tflite` and `labels.txt`. Send it to me and I will install it into the app (labels use offset 0, which the app auto-detects). The bundled app already ships a fresh copy of this focused classroom model + `labelmap.txt`, so if you do not retrain, the app works out of the box with the classroom set.